In [3]:
import os
import json
import random
import string
from typing import Annotated, TypedDict, List, Sequence, Optional, Dict
from operator import add
from langchain_groq import ChatGroq 
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command
from langchain_core.messages import BaseMessage,HumanMessage,SystemMessage,AIMessage,ToolMessage
from langgraph.graph.message import add_messages
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, InjectedState, tools_condition
from dotenv import load_dotenv

In [4]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [5]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [10]:
class SupervisorState(MessagesState):
    """
        State for multi-agent system
    """   
    user_input: str
    user_intent: str

    actions_taken: Annotated[list, add]
    observations: Annotated[list, add]
    mode: str
    complaint: dict
    missing_info: list
    next: str
    customer_profile: Dict
    

In [11]:
# Supervisor Node
def supervisor_node(state:SupervisorState)->SupervisorState:

    if state.get('mode')=='execute':
        return Command(goto="decision_node")
 
    system_prompt = """
    You are an intent classifier.

    Classify the user message into EXACTLY one of:
    - inquiry
    - complaint
    - churn
    - unknown

    Rules:
    - If the message is vague, emotional without clear intent, or lacks actionable meaning → return "unknown".
    - If multiple intents are possible and you are not confident → return "unknown".
    - Return ONLY label.

    Examples:
    "hello" → unknown
    "this is bad" → unknown
    "my order is damaged" → complaint
    "I want to cancel my subscription" → churn
    "What is the price?" → inquiry
    """
    
    response = llm.invoke([
        {"role":"system", "content":system_prompt},
        {"role":"user", "content":state["user_input"]}
        ])
    
    intent=response.content.strip().lower()
    
    print("User Intent is: ", intent)
    mode='execute'
    if intent =='unknown' and not state.get('mode'):
        mode='explore'
    
    return Command(goto="decision_node", update={
        "user_intent":intent,
        "mode":mode
    })


In [12]:
def decision_node(state:SupervisorState)->SupervisorState:

    mode = state.get('mode')
    intent=state.get('user_intent')

    print("Mode on: ", mode)
    print("Intent on: ", intent)

    if mode=="explore":
        allowed_actions=[
            "respond_greeting",
            "ask_open_question",
            "ask_clarification",
            "extract_info",
            "set_intent",
            "move_to_execute"
        ]
    else:
        if intent=='inquiry':
            allowed_actions=[
                "provide_info",
                END
            ]
        elif intent=='churn':
            allowed_actions=[
                "ask_clarification",
                "evaluate_user_value",
                "offer_retention_action",
                END
            ]
        elif intent=='complaint':
            allowed_actions=[
                "ask_clarification",
                "extract_info",
                "ask_missing_info",
                "create_ticket",
                END
            ]
        else:
            allowed_actions=["ask_clarification"]


    prompt=f"""
    You are a support agent.

    User: {state['user_input']}
    Intent: {state['user_intent']}
    mode: {state['mode']}
    Observations: {state['observations']}
    Actions taken: {state['actions_taken']}

    Choose appropriate Next action from:
    {allowed_actions}
    """

    decision = llm.invoke([{"role":"system", "content":prompt}]).content.strip()

    return Command(goto=decision)

In [13]:
# Worker nodes

def inquiry_node(state:SupervisorState)->SupervisorState:
    return {'messages':[f"Inquiry Handled: {state['user_input']}"]}

def fallback(state:SupervisorState)->SupervisorState:
    return {'messages':[f"Sorry, I couldn't understand your message"]}

In [14]:
def ask_clarification(state:SupervisorState)->SupervisorState:
    """
        It ask user follow up question to understand user query and better assist user.
    """
    user_input = state.get("user_input")
    prompt=f"""
        You are smart assistant. User said {user_input} earlier and you are trying to understand user better by 
        asking follow up questions to user to understand what user is looking for.
    """
    response = llm.invoke([
            SystemMessage(content=prompt),
            HumanMessage(content=user_input)
        ])
    return state

def extract_info_node(state:SupervisorState)->SupervisorState:
    """ 
        NLU Extractor agent that extract entities (product, issue_type, and purchase_date) from user input. 
    """
    user_input = state["user_input"]
    current_complaint = state.get('complaint',{})

    system_prompt=f""" You are an entity extractor. 
    Your goal is to extract the product information, issue_type, and purchase_date 
    from the user's input.
    - product
    - issue_type
    - purchase_date

    Respond ONLY with a valid JSON object in this exact format:
    {{
        "product": string or null,
        "issue_type": string or null,
        "purchase_date": MM/DD/YY or null
    }}
    """
     
    response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_input)
        ])
    
    complaint_data =json.loads(response.content)
    updated_complaint = {k: current_complaint.get(k) or v for k,v in complaint_data.items()}
    missing_data = [k for k,v in updated_complaint.items() if not v]
    state['complaint'] = updated_complaint
    state['missing_info']=missing_data
    
    return state
   
def ask_missing_info(state:SupervisorState)->SupervisorState:
    """" 
        This Agent validate th data from NLU agent and ask missing information from user input.
    """
    complaint_data = state['complaint']
    user_input = state['user_input']
    missing_data = state['missing_info']

    system_prommpt = f"""You are a smart assistant.
    This is a user input {user_input} and this is missing list of information {missing_data}.
    You will ask only missing info to user in concise and polite way.
    """
    response = llm.invoke([
        SystemMessage(content=system_prommpt)]
    )
    state['messages']=response.content
    return state

def create_ticket_node(state:SupervisorState)->SupervisorState:
    """
    Tool to create a complaint ticket.
    Generates a ticket ID like IG408C90.
    """

    complaint = state['complaint']
    
    prefix = ''.join(random.choices(string.ascii_uppercase,k=2))
    number1 = random.randint(100,1000)
    mid = ''.join(random.choices(string.ascii_uppercase))
    number2 = random.randint(10,100)
    ticket_id = f"{prefix}{number1}{mid}{number2}"

    ticket={
        "ticket_id":ticket_id,
        "status":"created",
        "details":complaint
    }
    state['messages']=f"Ticket {ticket['ticket_id']} has been created."
    
    return state

def router_node(state:SupervisorState)->SupervisorState:
    """Returns the next step key based on the missing_info list."""

    missing_info= state['missing_info']

    if missing_info:
        return "missing"
    else:
        return "no_missing"  

In [15]:
def churn_score_node(state:SupervisorState)->SupervisorState:
    """
        Calculate a churn risk score from user input.
        Returns category: high, medium, or low.
    """
    user_input=state['user_input']
    current_profile = state.get('customer_profile',{})
    system_prompt = """ 
    You are a churn detection assistant.
    Based on the user input, classifiy their churn risk into:
    - high: user explicitly wants to cancel, switch, or sounds very frustrated.
    - medium: user shows dissatisfaction but hasn't decided to cancel yet.
    - low: user just asking questions or mild complaints.

    Examples:
    User: "I'm cancelling this useless service today."
    Churn risk: high

    User: "Your prices keep going up, I don't know if it's worth it anymore."
    Churn risk: medium

    User: "How do I cancel if I ever need to in the future?"
    Churn risk: low

    Respond with only one of: high, medium, low.
    """
    score = llm.invoke([
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':user_input}
    ]).content
    
    current_profile['churn_score']=score
    return {"customer_profile":current_profile}

def loyalty_score_node(state:SupervisorState)->SupervisorState:
    """ 
        Provide loyalty score to user based on churn score and customer value. 
    """
    score = state['customer_profile']['churn_score']
    current_profile = state.get('customer_profile')
    reward_weights = {'high': 1.0, "medium": 0.6, "low": 0.2}
    clv_values = {"high": 1000, "medium": 500, "low": 200}
    clv_tier=random.choices(
        ['high','medium','low'],
        weights=[0.25,0.45,0.3],
        k=1
        )[0]

    loyalty_score = reward_weights[score]*clv_values[clv_tier]
    current_profile['loyalty_score']=loyalty_score
    current_profile['clv_tier']=clv_tier

    return {'customer_profile':current_profile}

def reward_node(state:SupervisorState)->SupervisorState:
    """
    Generate a personalized reward offers.
    """

    user_input=state['user_input']
    churn_score=state['customer_profile']['churn_score']
    loyalty_score=state['customer_profile']['loyalty_score']

    system_prompt=""" 
    You are a customer retention assistant. 
    Generate a polite, empathetic message based on the following:
    - The user's message
    - Their churn risk (high, medium, low)
    - Their loyalty score (40-1000)

    Rules:
    - If loyalty score >= 800 → emphasize strong appreciation and give a high reward (e.g., big discount, free premium month).
    - If 500-799 → show gratitude and offer a medium reward (e.g., discount or perk).
    - If 200-499 → acknowledge their value and give a small reward (e.g., loyalty points or small discount).
    - If < 200 → do not give a reward, just apologize and promise to improve.
    - If churn risk = high → always start by apologizing and showing empathy before mentioning any reward.
    - Keep the message short, friendly, concise and natural. Do not include technical terms or scores.
    
    Respond with only the final concise message.
    """

    user_context = f"""
        User message: {user_input}
        Churn risk: {churn_score}
        Loyalty score: {loyalty_score}
        """

    response = llm.invoke([{'role':'system', 'content':system_prompt},
                           {'role':'user','content':user_context}])
    state['messages']=response.content
    return state


In [16]:
graph = StateGraph(SupervisorState)

graph.add_node("supervisor", supervisor_node)
graph.add_node("decision_node", decision_node)
graph.add_node("ask_clarification", ask_clarification)
graph.add_node("extract_info", extract_info_node)
graph.add_node("ask_missing_info",ask_missing_info)
graph.add_node("create_ticket",create_ticket_node)

graph.add_node("Churn",churn_score_node)
graph.add_node("evaluate_user_value",loyalty_score_node)
graph.add_node("offer_retention_action",reward_node)

graph.add_node("inquiry",inquiry_node)

graph.add_edge(START, "supervisor")

memory=MemorySaver()
service_flow = graph.compile(checkpointer=memory)
config= {"configurable":{"thread_id":"user_complaint_session"}}

In [17]:
service_flow.invoke({"user_input":"My Laptop is broken and I just hate you"},config=config)

User Intent is:  complaint
Mode on:  execute
Intent on:  complaint


Task decision_node with path ('__pregel_pull', 'decision_node') wrote to unknown channel branch:to:I can see that you're upset about your laptop issue. I'm here to help and apologize if there's been any inconvenience caused.

Next action: ask_clarification, ignoring it.


{'messages': [],
 'user_input': 'My Laptop is broken and I just hate you',
 'user_intent': 'complaint',
 'actions_taken': [],
 'observations': [],
 'mode': 'execute'}